# Dar es Salaam Traffic EDA
Exploratory Data Analysis of historical telemetry data collected via the Dar Traffic Digital Twin.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set seaborn style for better aesthetics
sns.set_theme(style="darkgrid", context="notebook")

# Load data
df = pd.read_csv("historical_traffic_data.csv")
df.head()

## 1. Data Cleaning & Feature Engineering
Convert timestamps to the correct timezone and extract useful time-based features (Hour, Day).

In [ ]:
# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', utc=True).dt.tz_convert('Africa/Dar_es_Salaam')

# Extract features
df['hour'] = df['timestamp'].dt.hour
df['day_name'] = df['timestamp'].dt.day_name()
df['is_weekend'] = df['day_name'].isin(['Saturday', 'Sunday'])

# Clean weather labels
def map_weather(w):
    w = str(w).lower()
    if any(rain_word in w for rain_word in ["rain", "drizzle", "shower", "storm"]):
        return "Rainy"
    elif any(cloud_word in w for cloud_word in ["cloud", "overcast"]):
        return "Cloudy"
    else:
        return "Clear"

df['weather_clean'] = df['weather'].apply(map_weather)

df.info()

## 2. Average Delay by Time of Day
This reveals the "Rush Hour" peaks across the entire city network.

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='hour', y='delay_mins', estimator='mean', errorbar=None, marker='o')
plt.title('Average Network Delay by Hour of Day')
plt.xlabel('Hour (24h)')
plt.ylabel('Average Delay (Minutes)')
plt.xticks(range(0, 24))
plt.show()

## 3. The Worst Corridors
Identifying the arteries with the highest structural gridlock.

In [ ]:
road_delays = df.groupby('name')['delay_mins'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(12, 8))
sns.barplot(data=road_delays, x='delay_mins', y='name', palette='Reds_r')
plt.title('Average Delay by Artery')
plt.xlabel('Average Delay (Minutes)')
plt.ylabel('')
plt.tight_layout()
plt.show()

## 4. Meteorological Impact
How does rain affect average speeds across the network?

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='weather_clean', y='speed_kmh', palette=['#00D4FF', '#8892A4', '#FF4757'])
plt.title('Impact of Weather on Traffic Speed')
plt.xlabel('Condition')
plt.ylabel('Speed (km/h)')
plt.show()